In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

x_train = pd.read_csv(
    "x_train_final.csv",
    header=None,
    names=["idx", "idx2", "train", "gare", "date", "arret",
           "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"],
    skiprows=1
)
y_train = pd.read_csv("y_train_final.csv")

print(f"x_train : {x_train.shape[0]:,} lignes")

# =============================================================================
# 2. FEATURES
# =============================================================================

FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]

X = x_train[FEATURES].values
y = y_train["p0q0"].values

# =============================================================================
# 3. SPLIT 90 / 10
# =============================================================================

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

print(f"Train      : {X_tr.shape[0]:,} lignes")
print(f"Validation : {X_val.shape[0]:,} lignes")

# =============================================================================
# 4. RANDOM FOREST
# =============================================================================

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

# =============================================================================
# 5. ÉVALUATION
# =============================================================================

y_pred = rf.predict(X_val)

print(f"\nMAE  : {mean_absolute_error(y_val, y_pred):.4f}")

x_train : 667,264 lignes
Train      : 600,537 lignes
Validation : 66,727 lignes


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    9.0s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s



MAE  : 0.8741


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.2s finished


In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# =============================================================================
# 1. CHARGEMENT
# =============================================================================

x_train = pd.read_csv(
    "x_train_final.csv",
    header=None,
    names=["idx", "idx2", "train", "gare", "date", "arret",
           "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"],
    skiprows=1
)
y_train = pd.read_csv("y_train_final.csv")
x_test  = pd.read_csv("x_test_final.csv")

# =============================================================================
# 2. FEATURES
# =============================================================================

FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]

X      = x_train[FEATURES].values
y      = y_train["p0q0"].values
X_test = x_test[FEATURES].values

# =============================================================================
# 3. ENTRAÎNEMENT SUR 100% DU TRAIN
# =============================================================================

print("Entraînement sur 100% des données...")

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X, y)

# =============================================================================
# 4. PRÉDICTION ET SOUMISSION
# =============================================================================

y_pred = rf.predict(X_test)

# Arrondi à l'entier (la cible est en minutes entières)
y_pred_rounded = np.round(y_pred).astype(int)

submission = pd.DataFrame({
    "index": x_test["Unnamed: 0"],
    "p0q0":  y_pred_rounded
})

submission.to_csv("submission.csv", index=False)
print(f"submission.csv généré ({len(submission):,} lignes)")

Entraînement sur 100% des données...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    2.1s


submission.csv généré (20,657 lignes)


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   12.5s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.0s finished


In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

x_train = pd.read_csv(
    "x_train_final.csv",
    header=None,
    names=["idx", "idx2", "train", "gare", "date", "arret",
           "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"],
    skiprows=1
)
y_train = pd.read_csv("y_train_final.csv")

print(f"x_train : {x_train.shape[0]:,} lignes")

# =============================================================================
# 2. FEATURES
# =============================================================================

x_train["date"] = pd.to_datetime(x_train["date"])
jours_ohe = pd.get_dummies(x_train["date"].dt.dayofweek, prefix="jour")
 
FEATURES_NUM = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]
 
X = pd.concat([x_train[FEATURES_NUM], jours_ohe], axis=1).values
y = y_train["p0q0"].values

# =============================================================================
# 3. SPLIT 90 / 10
# =============================================================================

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

print(f"Train      : {X_tr.shape[0]:,} lignes")
print(f"Validation : {X_val.shape[0]:,} lignes")

# =============================================================================
# 4. RANDOM FOREST
# =============================================================================

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

# =============================================================================
# 5. ÉVALUATION
# =============================================================================

y_pred = rf.predict(X_val)

print(f"\nMAE  : {mean_absolute_error(y_val, y_pred):.4f}")

x_train : 667,264 lignes
Train      : 600,537 lignes
Validation : 66,727 lignes


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    2.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   14.7s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s



MAE  : 0.8926


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.4s finished


In [10]:
# Tendance du train (retards des trains précédents)
x_train["mean_retard_train"] = x_train[["p2q0","p3q0","p4q0"]].mean(axis=1)

# Tendance de la gare (retards aux gares précédentes)
x_train["mean_retard_gare"] = x_train[["p0q2","p0q3","p0q4"]].mean(axis=1)

# Retard total "contexte global" (moyenne de toutes les valeurs passées)
x_train["mean_retard_global"] = x_train[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].mean(axis=1)

In [11]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global"]

X = x_train[FEATURES].values
y = y_train["p0q0"].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(X_val)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    2.9s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   14.6s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s


MAE : 0.8653


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.2s finished


In [12]:
# Écart-type des retards (volatilité : est-ce que le train est régulièrement en retard ou c'est chaotique ?)
x_train["std_retard_train"]  = x_train[["p2q0","p3q0","p4q0"]].std(axis=1)
x_train["std_retard_gare"]   = x_train[["p0q2","p0q3","p0q4"]].std(axis=1)

# Tendance : est-ce que le retard s'aggrave ou s'améliore ?
x_train["tendance_train"] = x_train["p2q0"] - x_train["p4q0"]  # retard récent - retard ancien
x_train["tendance_gare"]  = x_train["p0q2"] - x_train["p0q4"]

# Retard maximum observé (cas extrêmes)
x_train["max_retard"] = x_train[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].max(axis=1)
x_train["min_retard"] = x_train[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].min(axis=1)

In [13]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard"]

X = x_train[FEATURES].values
y = y_train["p0q0"].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(X_val)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    4.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   23.3s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s


MAE : 0.8604


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.2s finished


In [14]:
from sklearn.preprocessing import LabelEncoder

le_train = LabelEncoder()
le_gare  = LabelEncoder()

x_train["train_enc"] = le_train.fit_transform(x_train["train"])
x_train["gare_enc"]  = le_gare.fit_transform(x_train["gare"])

In [15]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard",
            "train_enc", "gare_enc"]

X = x_train[FEATURES].values
y = y_train["p0q0"].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(X_val)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    7.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   36.3s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s


MAE : 0.7864


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.3s finished


In [16]:
stats_gare = x_train.join(y_train[["p0q0"]]).groupby("gare")["p0q0"].agg(
    gare_mean="mean",
    gare_std="std",
    gare_median="median",
    gare_q25=lambda x: x.quantile(0.25),
    gare_q75=lambda x: x.quantile(0.75)
).reset_index()

x_train = x_train.merge(stats_gare, on="gare", how="left")

In [17]:
stats_train = x_train.join(y_train[["p0q0"]]).groupby("train")["p0q0"].agg(
    train_mean="mean",
    train_std="std",
    train_median="median"
).reset_index()

x_train = x_train.merge(stats_train, on="train", how="left")

In [18]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard",
            "train_enc", "gare_enc",
            "gare_mean", "gare_std", "gare_median", "gare_q25", "gare_q75",
            "train_mean", "train_std", "train_median"]

X = x_train[FEATURES].values
y = y_train["p0q0"].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.10, random_state=42)

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(X_tr, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(X_val)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   10.7s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   54.2s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s


MAE : 0.6713


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.3s finished


In [19]:
# On repart de x_train propre (sans les colonnes stats déjà ajoutées)
cols_a_supprimer = ["gare_mean","gare_std","gare_median","gare_q25","gare_q75",
                    "train_mean","train_std","train_median"]
x_train = x_train.drop(columns=cols_a_supprimer)

# Split d'abord
X_idx = np.arange(len(x_train))
idx_tr, idx_val = train_test_split(X_idx, test_size=0.10, random_state=42)

train_fold = x_train.iloc[idx_tr].copy()
val_fold   = x_train.iloc[idx_val].copy()
y_tr       = y_train["p0q0"].iloc[idx_tr].values
y_val      = y_train["p0q0"].iloc[idx_val].values

# Stats calculées UNIQUEMENT sur les 90%
stats_gare = train_fold.join(y_train["p0q0"].iloc[idx_tr].rename("p0q0")).groupby("gare")["p0q0"].agg(
    gare_mean="mean", gare_std="std", gare_median="median",
    gare_q25=lambda x: x.quantile(0.25), gare_q75=lambda x: x.quantile(0.75)
).reset_index()

stats_train = train_fold.join(y_train["p0q0"].iloc[idx_tr].rename("p0q0")).groupby("train")["p0q0"].agg(
    train_mean="mean", train_std="std", train_median="median"
).reset_index()

# On applique sur train ET val (le val utilise les stats du train → pas de fuite)
train_fold = train_fold.merge(stats_gare, on="gare", how="left")
train_fold = train_fold.merge(stats_train, on="train", how="left")

val_fold = val_fold.merge(stats_gare, on="gare", how="left")
val_fold = val_fold.merge(stats_train, on="train", how="left")

In [20]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard",
            "train_enc", "gare_enc",
            "gare_mean", "gare_std", "gare_median", "gare_q25", "gare_q75",
            "train_mean", "train_std", "train_median"]

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(train_fold[FEATURES].values, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(val_fold[FEATURES].values)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   10.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   55.7s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s


MAE : 0.7518


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.3s finished


In [21]:
import networkx as nx

G = nx.DiGraph()

for (train, date), groupe in x_train.groupby(["train", "date"]):
    groupe = groupe.sort_values("arret")
    gares  = groupe["gare"].tolist()

    for i in range(len(gares)):
        G.add_node(gares[i])

        if i > 0:
            G.add_edge(gares[i-1], gares[i]) # ajouter retard et q0p0, trouver retard moyen entre deux gares

print(f"Noeuds (gares) : {G.number_of_nodes()}")
print(f"Arêtes (liaisons) : {G.number_of_edges()}")

Noeuds (gares) : 84
Arêtes (liaisons) : 709


In [22]:
# Rang relatif de la gare dans le trajet (0 = départ, 1 = terminus)
position_gare = {}
for (train, date), groupe in x_train.groupby(["train", "date"]):
    groupe = groupe.sort_values("arret")
    n = len(groupe)
    for i, (_, row) in enumerate(groupe.iterrows()):
        position_gare[row["gare"]] = position_gare.get(row["gare"], [])
        position_gare[row["gare"]].append(i / (n - 1) if n > 1 else 0)

position_df = pd.DataFrame({
    "gare": list(position_gare.keys()),
    "position_moyenne": [np.mean(v) for v in position_gare.values()]
})

train_fold = train_fold.merge(position_df, on="gare", how="left")
val_fold   = val_fold.merge(position_df, on="gare", how="left")

In [23]:
freq_gare = x_train.groupby(["gare", "date"])["train"].nunique().groupby("gare").mean().reset_index()
freq_gare.columns = ["gare", "freq_trains_par_jour"]

train_fold = train_fold.merge(freq_gare, on="gare", how="left")
val_fold   = val_fold.merge(freq_gare, on="gare", how="left")

In [24]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard",
            "train_enc", "gare_enc",
            "gare_mean", "gare_std", "gare_median", "gare_q25", "gare_q75",
            "train_mean", "train_std", "train_median",
            "position_moyenne", "freq_trains_par_jour"]

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(train_fold[FEATURES].values, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(val_fold[FEATURES].values)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   11.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   58.1s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s


MAE : 0.7505


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.2s finished


In [25]:
# Fusion temporaire pour avoir la date et le target ensemble
train_with_target = train_fold.copy()
train_with_target["p0q0"] = y_tr
train_with_target["date"] = pd.to_datetime(train_with_target["date"])

def stats_7j(df, group_col):
    records = []
    for date in df["date"].unique():
        date_min = date - pd.Timedelta(days=7)
        fenetre  = df[(df["date"] > date_min) & (df["date"] < date)]
        stats    = fenetre.groupby(group_col)["p0q0"].agg(
            **{f"{group_col}_mean_7j": "mean",
               f"{group_col}_std_7j":  "std"}
        ).reset_index()
        stats["date"] = date
        records.append(stats)
    return pd.concat(records, ignore_index=True)

stats_train_7j = stats_7j(train_with_target, "train")
stats_gare_7j  = stats_7j(train_with_target, "gare")

# Merge sur train_fold et val_fold
train_fold["date"] = pd.to_datetime(train_fold["date"])
val_fold["date"]   = pd.to_datetime(val_fold["date"])

train_fold = train_fold.merge(stats_train_7j, on=["train", "date"], how="left")
train_fold = train_fold.merge(stats_gare_7j,  on=["gare",  "date"], how="left")

val_fold = val_fold.merge(stats_train_7j, on=["train", "date"], how="left")
val_fold = val_fold.merge(stats_gare_7j,  on=["gare",  "date"], how="left")

# Valeurs manquantes (train/gare sans historique sur 7j) remplacées par la moyenne globale
train_fold["train_mean_7j"] = train_fold["train_mean_7j"].fillna(train_fold["train_mean"])
train_fold["train_std_7j"]  = train_fold["train_std_7j"].fillna(train_fold["train_std"])
train_fold["gare_mean_7j"]  = train_fold["gare_mean_7j"].fillna(train_fold["gare_mean"])
train_fold["gare_std_7j"]   = train_fold["gare_std_7j"].fillna(train_fold["gare_std"])

val_fold["train_mean_7j"] = val_fold["train_mean_7j"].fillna(val_fold["train_mean"])
val_fold["train_std_7j"]  = val_fold["train_std_7j"].fillna(val_fold["train_std"])
val_fold["gare_mean_7j"]  = val_fold["gare_mean_7j"].fillna(val_fold["gare_mean"])
val_fold["gare_std_7j"]   = val_fold["gare_std_7j"].fillna(val_fold["gare_std"])

In [26]:
FEATURES = ["arret", "p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4",
            "mean_retard_train", "mean_retard_gare", "mean_retard_global",
            "std_retard_train", "std_retard_gare",
            "tendance_train", "tendance_gare",
            "max_retard", "min_retard",
            "train_enc", "gare_enc",
            "gare_mean", "gare_std",
            "train_mean", "train_std", "train_median",
            "position_moyenne", "freq_trains_par_jour",
            "train_mean_7j", "train_std_7j",
            "gare_mean_7j",  "gare_std_7j"]

rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, verbose=1)
rf.fit(train_fold[FEATURES].values, y_tr)

print(f"MAE : {mean_absolute_error(y_val, rf.predict(val_fold[FEATURES].values)):.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   14.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.3min finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s


MAE : 0.7467


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.3s finished


In [27]:
x_test = pd.read_csv("x_test_final.csv")
x_test["date"] = pd.to_datetime(x_test["date"])

# Features de base
x_test["mean_retard_train"]  = x_test[["p2q0","p3q0","p4q0"]].mean(axis=1)
x_test["mean_retard_gare"]   = x_test[["p0q2","p0q3","p0q4"]].mean(axis=1)
x_test["mean_retard_global"] = x_test[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].mean(axis=1)
x_test["std_retard_train"]   = x_test[["p2q0","p3q0","p4q0"]].std(axis=1)
x_test["std_retard_gare"]    = x_test[["p0q2","p0q3","p0q4"]].std(axis=1)
x_test["tendance_train"]     = x_test["p2q0"] - x_test["p4q0"]
x_test["tendance_gare"]      = x_test["p0q2"] - x_test["p0q4"]
x_test["max_retard"]         = x_test[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].max(axis=1)
x_test["min_retard"]         = x_test[["p2q0","p3q0","p4q0","p0q2","p0q3","p0q4"]].min(axis=1)

# Encodage train & gare
# Encodage robuste aux valeurs inconnues
train_mapping = {v: i for i, v in enumerate(le_train.classes_)}
gare_mapping  = {v: i for i, v in enumerate(le_gare.classes_)}

x_test["train_enc"] = x_test["train"].map(train_mapping).fillna(-1).astype(int)
x_test["gare_enc"]  = x_test["gare"].map(gare_mapping).fillna(-1).astype(int)

# Stats globales (calculées sur train_fold)
x_test = x_test.merge(stats_gare,  on="gare",  how="left")
x_test = x_test.merge(stats_train, on="train", how="left")

# Position & fréquence
x_test = x_test.merge(position_df, on="gare", how="left")
x_test = x_test.merge(freq_gare,   on="gare", how="left")

# Stats 7 jours
x_test = x_test.merge(stats_train_7j, on=["train", "date"], how="left")
x_test = x_test.merge(stats_gare_7j,  on=["gare",  "date"], how="left")

# Fallback valeurs manquantes
x_test["train_mean_7j"] = x_test["train_mean_7j"].fillna(x_test["train_mean"])
x_test["train_std_7j"]  = x_test["train_std_7j"].fillna(x_test["train_std"])
x_test["gare_mean_7j"]  = x_test["gare_mean_7j"].fillna(x_test["gare_mean"])
x_test["gare_std_7j"]   = x_test["gare_std_7j"].fillna(x_test["gare_std"])

# Prédiction
y_pred = rf.predict(x_test[FEATURES].values)
y_pred_rounded = np.round(y_pred).astype(int)

submission = pd.DataFrame({
    "index": x_test["Unnamed: 0"],
    "p0q0":  y_pred_rounded
})

submission.to_csv("submission2+.csv", index=False)
print(f"submission+.csv généré ({len(submission):,} lignes)")

[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.


submission+.csv généré (20,657 lignes)


[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.0s finished


In [28]:
importances = pd.Series(rf.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=False)
print(importances.to_string())

gare_mean_7j            0.143452
train_mean_7j           0.085343
train_mean              0.083368
train_std_7j            0.058034
train_std               0.057798
arret                   0.048648
gare_std_7j             0.044015
mean_retard_gare        0.041541
train_enc               0.039016
std_retard_train        0.034812
gare_mean               0.032185
mean_retard_train       0.027735
p0q2                    0.023894
gare_enc                0.023777
std_retard_gare         0.023762
mean_retard_global      0.022215
gare_std                0.020690
tendance_gare           0.018973
p0q3                    0.018518
p2q0                    0.018427
p0q4                    0.017887
position_moyenne        0.017860
train_median            0.016643
freq_trains_par_jour    0.016065
p3q0                    0.015694
tendance_train          0.014856
p4q0                    0.013708
min_retard              0.012760
max_retard              0.008323


In [42]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras, torch
from keras.layers import Dense, BatchNormalization, Dropout, Input
from sklearn.preprocessing import RobustScaler, StandardScaler

print(f"Backend: {keras.backend.backend()}, CUDA: {torch.cuda.is_available()}")

# ===== Features SANS les 5 colonnes NaN dans le test (99.99% inconnus) =====
NN_FEATURES = [f for f in FEATURES if f not in
               ["train_mean", "train_std", "train_median", "train_mean_7j", "train_std_7j"]]
print(f"Features retenues : {len(NN_FEATURES)} (retirées: train_mean/std/median/7j)")

X_nn_tr  = train_fold[NN_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)
X_nn_val = val_fold[NN_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)

# Scaler X
scaler_X = RobustScaler()
X_nn_tr  = np.clip(scaler_X.fit_transform(X_nn_tr), -10, 10).astype(np.float32)
X_nn_val = np.clip(scaler_X.transform(X_nn_val), -10, 10).astype(np.float32)

# Scaler y (CRUCIAL)
scaler_y = StandardScaler()
y_tr_s = scaler_y.fit_transform(y_tr.reshape(-1, 1)).astype(np.float32)
y_val_s = scaler_y.transform(y_val.reshape(-1, 1)).astype(np.float32)

print(f"y range: [{y_tr_s.min():.1f}, {y_tr_s.max():.1f}]")

# ===== MLP simple =====
n = len(NN_FEATURES)
inp = Input(shape=(n,))
x = Dense(256, activation="relu")(inp)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
x = Dense(64, activation="relu")(x)
x = BatchNormalization()(x)
x = Dense(32, activation="relu")(x)
out = Dense(1)(x)

model_nn = keras.Model(inp, out)
model_nn.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras.losses.Huber(delta=1.0), metrics=["mae"])
model_nn.summary()

history = model_nn.fit(
    X_nn_tr, y_tr_s,
    validation_data=(X_nn_val, y_val_s),
    epochs=200, batch_size=512,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau("val_mae", factor=0.5, patience=10, min_lr=1e-6, verbose=1),
        keras.callbacks.EarlyStopping("val_mae", patience=30, restore_best_weights=True, verbose=1)
    ],
    verbose=1
)

# ===== MAE réelle =====
pred_nn_val = scaler_y.inverse_transform(model_nn.predict(X_nn_val)).flatten()
pred_rf_val = rf.predict(val_fold[FEATURES].values)
mae_nn = mean_absolute_error(y_val, pred_nn_val)
mae_rf = mean_absolute_error(y_val, pred_rf_val)

print(f"\n{'='*40}")
print(f"MAE RF:  {mae_rf:.4f}")
print(f"MAE NN:  {mae_nn:.4f}")
print(f"{'='*40}")

Backend: tensorflow, CUDA: True
Features retenues : 24 (retirées: train_mean/std/median/7j)
y range: [-80.9, 7.7]


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 24)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 256)            │         6,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_23          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_24          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,457 (201.00 KB)

 Trainable params: 50,561 (197.50 KB)

 Non-trainable params: 896 (3.50 KB)

Epoch 1/200
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.1645 - mae: 0.4125 - val_loss: 0.1484 - val_mae: 0.3835 - learning_rate: 0.0010
Epoch 2/200
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.1510 - mae: 0.3878 - val_loss: 0.1440 - val_mae: 0.3728 - learning_rate: 0.0010
Epoch 3/200
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.1470 - mae: 0.3804 - val_loss: 0.1408 - val_mae: 0.3694 - learning_rate: 0.0010
Epoch 4/200
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.1447 - mae: 0.3759 - val_loss: 0.1386 - val_mae: 0.3628 - learning_rate: 0.0010
Epoch 5/200
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.1430 - mae: 0.3726 - val_loss: 0.1372 - val_mae: 0.3606 - learning_rate: 0.0010
Epoch 6/200
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.1419 - mae: 0.3708 - val_loss: 0.1364 - val_mae: 0.3606 - learning_rate: 0.0010
Epoch 7/200
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.1409 - mae: 0.3690 - val_loss: 0.1361 - val_mae: 0.3603 - learnin

[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.4s



MAE RF:  0.7467
MAE NN:  0.6851


[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    2.9s finished


In [43]:
# ===== Soumission NN (24 features propres) + Ensemble RF+NN =====

# Prédiction test avec model_nn (early stopped, 24 features)
X_test_nn = x_test[NN_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)
X_test_nn_s = np.clip(scaler_X.transform(X_test_nn), -10, 10).astype(np.float32)

# Check val
pred_check = scaler_y.inverse_transform(model_nn.predict(X_nn_val)).flatten()
print(f"Check MAE val NN : {mean_absolute_error(y_val, pred_check):.4f}")

# NaN check
nan_test = x_test[NN_FEATURES].isna().sum().sum()
print(f"NaN dans x_test[NN_FEATURES] : {nan_test} (devrait être 0 ou quasi)")

# Prédiction NN
pred_nn_test = scaler_y.inverse_transform(model_nn.predict(X_test_nn_s)).flatten()

# Prédiction RF (déjà calculée dans cell 22, mais on la refait ici)
pred_rf_test = rf.predict(x_test[FEATURES].values)

# === Soumission NN pure ===
submission_nn = pd.DataFrame({"index": x_test["Unnamed: 0"], "p0q0": np.round(pred_nn_test).astype(int)})
submission_nn.to_csv("submission_nn_final.csv", index=False)
print(f"\n--- NN pur ---")
print(f"Stats: mean={pred_nn_test.mean():.2f}, std={pred_nn_test.std():.2f}, min={pred_nn_test.min():.2f}, max={pred_nn_test.max():.2f}")

# === Soumission Ensemble RF + NN ===
for alpha in [0.3, 0.5, 0.7]:
    pred_ens = alpha * pred_nn_test + (1 - alpha) * pred_rf_test
    sub = pd.DataFrame({"index": x_test["Unnamed: 0"], "p0q0": np.round(pred_ens).astype(int)})
    sub.to_csv(f"submission_ensemble_{int(alpha*100)}nn.csv", index=False)
    print(f"\n--- Ensemble {int(alpha*100)}%NN + {int((1-alpha)*100)}%RF ---")
    print(f"Stats: mean={pred_ens.mean():.2f}, std={pred_ens.std():.2f}")

# Ensemble optimal basé sur la val
best_alpha, best_mae = 0, 999
for a in np.arange(0, 1.01, 0.05):
    mix = a * pred_nn_val + (1 - a) * pred_rf_val
    m = mean_absolute_error(y_val, mix)
    if m < best_mae:
        best_alpha, best_mae = a, m
print(f"\n--- Meilleur alpha val: {best_alpha:.2f} (MAE val={best_mae:.4f}) ---")

pred_best = best_alpha * pred_nn_test + (1 - best_alpha) * pred_rf_test
sub_best = pd.DataFrame({"index": x_test["Unnamed: 0"], "p0q0": np.round(pred_best).astype(int)})
sub_best.to_csv("submission_best_ensemble.csv", index=False)
print(f"submission_best_ensemble.csv: mean={pred_best.mean():.2f}, std={pred_best.std():.2f}")

2086/2086 ━━━━━━━━━━━━━━━━━━━━ 1s 454us/step
Check MAE val NN : 0.6851
NaN dans x_test[NN_FEATURES] : 0 (devrait être 0 ou quasi)
646/646 ━━━━━━━━━━━━━━━━━━━━ 0s 472us/step

--- NN pur ---
Stats: mean=-0.14, std=0.91, min=-16.81, max=2.75

--- Ensemble 30%NN + 70%RF ---
Stats: mean=-0.11, std=0.80

--- Ensemble 50%NN + 50%RF ---
Stats: mean=-0.12, std=0.82

--- Ensemble 70%NN + 30%RF ---
Stats: mean=-0.12, std=0.85

--- Meilleur alpha val: 0.80 (MAE val=0.6796) ---
submission_best_ensemble.csv: mean=-0.13, std=0.87


[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.0s finished
